![image.png](https://i.imgur.com/a3uAqnb.png)


# 🧠 Attention-Based Transformer Classifier: IMDb Sentiment Analysis

This notebook demonstrates how to fine-tune a pretrained Transformer model (specifically `distilbert-base-uncased`) for binary text classification using the IMDb movie reviews dataset. The workflow follows a clean and modular structure, making it easy to extend or adapt to other NLP tasks.

### 📌 Objectives:
- Load and preprocess a real-world text classification dataset.
- Use Hugging Face’s 🤗 Transformers and Datasets libraries for efficient NLP modeling.
- Define a custom `TextDataset` for tokenized inputs.
- Fine-tune a pretrained Transformer model using the Hugging Face `Trainer` API.
- Track training progress and visualize evaluation metrics.
- Perform inference on custom inputs and interpret predictions.

### 🧰 Key Features:
- Uses the IMDb dataset with clear train/val/test splits.
- Implements custom configuration and logging.
- Includes early stopping and metric tracking (F1, Accuracy, etc.).
- Easy-to-use inference function for testing new text samples.

By the end of this notebook, you will have a fully trained binary sentiment classifier ready to predict whether a given movie review is **positive** or **negative**.


### 🧱 Imports and Setup

This cell imports all the necessary libraries for model development, including:
- **PyTorch** and **Transformers** for deep learning and NLP tasks.
- **Datasets** from Hugging Face to load datasets.
- **Scikit-learn** for evaluation metrics.
- **Matplotlib** and **Seaborn** for visualization.
- **TQDM** for progress bars.
- **Warnings** are suppressed for a cleaner output.

Additionally:
- Random seeds are set for reproducibility.
- Output directories are created to store trained models and logs.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import load_dataset
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Create output directory if it doesn't exist
import os
os.makedirs("./fine_tuned_model", exist_ok=True)
os.makedirs("./fine_tuned_model/logs", exist_ok=True)

### ⚙️ Configuration Parameters

Defines a `Config` class that centralizes key parameters for model training and evaluation:
- **Model settings**: Pretrained model name, maximum token length, number of labels.
- **Training settings**: Batch size, learning rate, number of epochs, weight decay, warmup ratio.
- **Data settings**: Proportions for train, validation, and test splits.
- **Output settings**: Directory paths and logging frequency.

Reviews are cut to 256 tokens and trained for 2 epochs, in 16-bit mixed precision when a GPU is present, which keeps the run short on a free Colab T4. `N_TRAIN` and `N_TEST` shrink the data for a quick test run.

This structure allows easy modification and readability.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

N_TRAIN = None   # None: the full IMDb training split (90% train, 10% validation); a number trains faster
N_TEST = None    # None: all 25,000 test reviews

class Config:
    """Configuration class for fine-tuning parameters"""

    # Model settings
    MODEL_NAME = "distilbert/distilbert-base-uncased"
    MAX_LENGTH = 256          # tokens per review; longer reviews are cut
    NUM_LABELS = 2  # Adjust based on your task

    # Training settings
    BATCH_SIZE = 32
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 2
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01

    # Output settings
    OUTPUT_DIR = "./fine_tuned_model"
    LOGGING_STEPS = 50

config = Config()
print(f"✅ Configuration loaded - Model: {config.MODEL_NAME}")

### 📦 Custom TextDataset Class

Defines a `TextDataset` class for tokenizing and formatting the raw text data into a format compatible with Hugging Face transformers:
- Uses the `tokenizer` to convert texts into `input_ids` and `attention_mask`.
- Returns tokenized inputs and the corresponding label tensors.
- Applies padding and truncation to a fixed max length.


In [ ]:
# =============================================================================
# DATA PREPARATION
# =============================================================================

class TextDataset(Dataset):
    """Custom dataset class for text classification"""
    
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Tokenize text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


### 🤖 Setup Model and Tokenizer

Initializes the tokenizer and model:
- Loads a pretrained model and tokenizer using Hugging Face's Transformers library.
- Adds a padding token if it's missing.
- Loads the classification model with the specified number of labels.
- Moves the model to the appropriate device (CPU/GPU).


In [ ]:
# =========================================================================|====
# MODEL SETUP
# =============================================================================

def setup_model_and_tokenizer():
    """Initialize tokenizer and model"""
    print(f"🤖 Loading model and tokenizer: {config.MODEL_NAME}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    
    # Add padding token if it doesn't exist
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load model for sequence classification
    model = AutoModelForSequenceClassification.from_pretrained(
        config.MODEL_NAME,
        num_labels=config.NUM_LABELS
    )
    
    # Move to GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    print(f"✅ Model loaded on device: {device}")
    return tokenizer, model, device



### 📥 Load IMDb Dataset and Prepare Splits

- Loads the IMDb dataset (binary sentiment classification) using Hugging Face `load_dataset`.
- Tokenizes and converts train/test data into `TextDataset` objects.
- Splits the training data into training and validation subsets.
- Also initializes the model, tokenizer, and sets up the device.


In [ ]:
raw_dataset = load_dataset("stanfordnlp/imdb")
train_split = raw_dataset['train'].shuffle(seed=42)
test_split = raw_dataset['test']
if N_TRAIN is not None:
    train_split = train_split.select(range(N_TRAIN))
if N_TEST is not None:
    test_split = test_split.shuffle(seed=42).select(range(N_TEST))

train_texts, train_labels = list(train_split['text']), list(train_split['label'])
test_texts, test_labels = list(test_split['text']), list(test_split['label'])

print(f"✅ Data loaded - Train samples: {len(train_texts)}, Test samples: {len(test_texts)}")

# Create tokenizer and model
tokenizer, model, device = setup_model_and_tokenizer()

# Create datasets
print("🔄 Creating datasets...")
train_dataset = TextDataset(train_texts, train_labels, tokenizer, config.MAX_LENGTH)
test_dataset = TextDataset(test_texts, test_labels, tokenizer, config.MAX_LENGTH)

# Split train into train/validation
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

print(f"✅ Datasets created - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

### 🧠 Trainer Setup and Evaluation Metrics

- Defines a `compute_metrics` function to calculate accuracy, precision, recall, and F1 score.
- Initializes Hugging Face `TrainingArguments` to configure:
  - Epochs, batch sizes, learning rate, logging frequency, evaluation once per epoch, and early stopping.
  - 16-bit mixed precision (`fp16`) when a GPU is present, which roughly halves memory and speeds up training on a T4.
- Sets up the `Trainer` class with:
  - Model, training/validation datasets, evaluation metrics, and early stopping callback.

In [ ]:
# =============================================================================
# TRAINING SETUP
# =============================================================================

def compute_metrics(eval_pred):
    """Compute metrics for evaluation"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    learning_rate=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    warmup_steps=config.WARMUP_RATIO,   # a value below 1 is read as a share of all training steps
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=config.LOGGING_STEPS,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=[],
)


# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] #based on F1 score aka 'metric_for_best_model'
)

print("✅ Training setup complete!")

### 🚀 Train the Model

- Triggers the training process using `trainer.train()`.
- Saves the fine-tuned model and tokenizer to the specified output directory.


In [ ]:
# =============================================================================
# TRAINING
# =============================================================================

def train_model():
    """Train the model"""
    print("🚀 Starting training...")
    
    # Train
    trainer.train()
    
    # Save final model
    trainer.save_model(config.OUTPUT_DIR)
    tokenizer.save_pretrained(config.OUTPUT_DIR)
    
    print(f"✅ Training complete! Model saved to {config.OUTPUT_DIR}")


train_model()

### 📊 Training Curves Visualization

- Extracts log history from the `Trainer` state.
- Plots:
  - **Training & Evaluation Loss** over steps.
  - **Evaluation Accuracy** over steps.
- Helps in diagnosing underfitting/overfitting and tracking training progress.


In [ ]:
# Extract logs
log_history = trainer.state.log_history

train_steps, train_loss = [], []
eval_steps, eval_loss, eval_accuracy = [], [], []
for log in log_history:
    if "loss" in log:
        train_steps.append(log["step"]); train_loss.append(log["loss"])
    if "eval_loss" in log:
        eval_steps.append(log["step"]); eval_loss.append(log["eval_loss"]); eval_accuracy.append(log["eval_accuracy"])

# Plotting
plt.figure(figsize=(12, 5))

# 📉 Training & Eval Loss
plt.subplot(1, 2, 1)
plt.plot(train_steps, train_loss, label='Train Loss')
plt.plot(eval_steps, eval_loss, marker='o', label='Eval Loss')
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training & Evaluation Loss")
plt.legend()

# ✅ Evaluation Accuracy
plt.subplot(1, 2, 2)
plt.plot(eval_steps, eval_accuracy, marker='o', label='Eval Accuracy', color='green')
plt.xlabel("Steps")
plt.ylabel("Accuracy")
plt.title("Evaluation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

### ✅ Evaluate on Test Set

- Evaluates the trained model on the test dataset using `trainer.evaluate`.
- Prints out final evaluation metrics such as accuracy, loss, and F1 score.


In [ ]:
# =============================================================================
# EVALUATION
# =============================================================================
print("📊 Evaluating model...")
    
# Evaluate on test set
test_results = trainer.evaluate(test_dataset)
    
print("Test Results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")
    

### 🔍 Predict on a Single Input Text

Defines a utility function `predict_single_text` to:
- Tokenize a single input string.
- Run the model in evaluation mode (no gradients).
- Apply softmax to get confidence scores.
- Return the predicted class and its confidence.


In [ ]:
def predict_single_text(text, model, tokenizer, device):
    """Make prediction on a single text"""
    model.eval()
    
    # Tokenize
    inputs = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=config.MAX_LENGTH,
        return_tensors='pt'
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(predictions, dim=1).item()
        confidence = predictions[0][predicted_class].item()
    
    return predicted_class, confidence

### 🔮 Inference on Example Sentences

- Provides a few example review sentences to demonstrate the inference capability of the model.
- Uses the `predict_single_text` function to classify each as Positive or Negative.
- Displays the prediction along with confidence scores.


In [ ]:
# =============================================================================
# INFERENCE EXAMPLES
# =============================================================================

def run_inference_examples():
    """Run some inference examples"""
    print("🔮 Running inference examples...")
    
    sample_texts = [
        "This product is amazing! I love it so much.",
        "Terrible quality, would not recommend to anyone.",
        "It's okay, nothing special but does the job.",
        "Outstanding performance and great value for money!"
    ]
    
    for text in sample_texts:
        pred_class, confidence = predict_single_text(text, model, tokenizer, device)
        sentiment = "Positive" if pred_class == 1 else "Negative"
        print(f"Text: '{text[:50]}...'")
        print(f"Prediction: {sentiment} (confidence: {confidence:.3f})\n")

### ▶️ Run Inference Examples

- Calls the `run_inference_examples()` function to demonstrate the model’s prediction behavior in action.
- Useful for quick manual inspection of how well the model understands sentiment.


In [ ]:
# Run example inference with the fine-tuned model
run_inference_examples()

### 📋 Comparing with Other Models on the Same Test Set

The IMDb test split has 25,000 reviews, and the fine-tuned model was scored on all of them. Any other
classifier scored on the same split can be compared with it directly. If you trained such models (for
example a recurrent network or a GRU with attention), type their test accuracies and parameter counts
into `other_models`; leave `None` for models you did not train.

In [ ]:
other_models = {
    # name: (test accuracy, parameters)
    "RNN": (None, None),
    "LSTM": (None, None),
    "GRU": (None, None),
    "GRU + attention": (None, None),
}

rows = [{"model": name, "test accuracy": acc, "parameters": params}
        for name, (acc, params) in other_models.items()]
rows.append({"model": "DistilBERT, fine-tuned",
             "test accuracy": round(test_results["eval_accuracy"], 4),
             "parameters": sum(p.numel() for p in model.parameters())})
comparison = pd.DataFrame(rows)
comparison

### Contributed by: Yazan Alshoibi